<a href="https://colab.research.google.com/github/amosagekouassi-source/DI-Bootcamp/blob/master/Mini_Projet_W9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MCP + Agents AI Integration in Gemini
This notebook demonstrates an end-to-end agentic application orchestrating multiple MCP servers (filesystem, git, and custom tools) using Gemini.

In [ ]:
%%capture
# 1. Install dependencies
%pip install -qU \
  "langchain>=0.3" \
  "langgraph>=0.2" \
  "langchain-google-genai>=2.0" \
  "google-genai>=1.0" \
  "langchain-mcp-adapters==0.2.1" \
  "nest_asyncio" \
  "fastmcp>=2.0.0"

In [ ]:
# 2. Setup Environment
import nest_asyncio
nest_asyncio.apply()

# Confirm Node/NPM availability
!node --version
!npx --version

## Custom MCP Server
We create a local Python server using `FastMCP` to provide specialized tools for our agent.

In [ ]:
from pathlib import Path
import textwrap

server_path = Path("/content/custom_mcp_server.py")
server_path.write_text(textwrap.dedent("""
    from fastmcp import FastMCP
    from typing import Dict, List

    mcp = FastMCP(name="custom_ops")

    @mcp.tool
    def ping() -> str:
        \"\"\"Health check tool.\"\"\"
        return "pong"

    @mcp.tool
    def summarize_lines(lines: List[str]) -> Dict[str, int]:
        \"\"\"Returns counts about a list of lines.\"\"\"
        total = len(lines)
        nonempty = sum(1 for l in lines if l.strip())
        return {"total_lines": total, "nonempty_lines": nonempty}

    if __name__ == "__main__":
        mcp.run(transport="stdio")
"""), encoding="utf-8")

print("Wrote custom server to:", server_path)